# Hunar M2: train the PPE / tool presence detector (YOLOv8n) on Google Colab

**Run on Colab with a GPU** (Runtime → Change runtime type → T4/L4/A100). Inference later runs on CPU in the app.

**What this model does and does not do.** It detects whether PPE (helmet, gloves, safety shoes, goggles) and, once the team adds labelled photos, electrician tools are *visible* in a task photo. It does **not** judge the quality of the work. Every number this notebook prints must be copied to the repo only through the generated `m2_detector.json`, never typed by hand.

**Data.** *Personal Protective Equipment* dataset (Roboflow export v7, 11,978 images, **CC BY 4.0**), mirrored on Hugging Face as `keremberke/protective-equipment-detection`. Attribution is in `docs/DATA.md`. The dataset contains many near-identical video frames, so step 4 measures train/test leakage and reports mAP on a *clean* test subset too.

In [ ]:
!nvidia-smi -L || echo "No GPU: training will be very slow"
%pip -q install "ultralytics==8.3.228" imagehash
import ultralytics; ultralytics.checks()

## 0. Keep the training run safe (recommended)

Colab can disconnect. Saving runs to Google Drive means the weights survive; if it disconnects,
re-run cells 0 to 2 and then the **resume** cell in step 5 instead of starting again.

In [ ]:
USE_DRIVE = True   # set False to keep everything in the temporary /content disk
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    RUNS = "/content/drive/MyDrive/hunar_runs"
else:
    RUNS = "/content/runs"
print("runs will be saved in", RUNS)

## 1. Download the dataset (train / valid / test, COCO format)

In [ ]:
import os, zipfile, urllib.request, json, hashlib, datetime, pathlib
BASE = "https://huggingface.co/datasets/keremberke/protective-equipment-detection/resolve/main/data/"
ROOT = pathlib.Path("/content/ppe"); ROOT.mkdir(exist_ok=True)
for split in ["train", "valid", "test"]:
    z = ROOT / f"{split}.zip"
    if not z.exists():
        urllib.request.urlretrieve(BASE + f"{split}.zip", z)
    if not (ROOT / split).exists():
        zipfile.ZipFile(z).extractall(ROOT / split)
    print(split, sum(1 for _ in (ROOT / split).glob("*.jpg")), "images")

## 2. Convert COCO → YOLO with Hunar class names

Mapping: `helmet→helmet`, `glove→gloves`, `shoes→safety_shoes`, `goggles→goggles`. The `no_*`, `mask` classes are dropped (Hunar only needs presence of the required item). The dataset has no high-visibility vests, so `vest` is not trained.

In [ ]:
CLASS_MAP = {"helmet": "helmet", "glove": "gloves", "shoes": "safety_shoes", "goggles": "goggles"}
NAMES = ["helmet", "gloves", "safety_shoes", "goggles"]
YOLO = pathlib.Path("/content/yolo"); YOLO.mkdir(exist_ok=True)

def convert(split):
    src = ROOT / split
    coco = json.load(open(src / "_annotations.coco.json"))
    cats = {c["id"]: c["name"] for c in coco["categories"]}
    imgs = {i["id"]: i for i in coco["images"]}
    labels = {i: [] for i in imgs}
    for a in coco["annotations"]:
        name = CLASS_MAP.get(cats[a["category_id"]])
        if name is None:
            continue
        im = imgs[a["image_id"]]
        x, y, w, h = a["bbox"]
        labels[a["image_id"]].append(f"{NAMES.index(name)} {(x + w/2)/im['width']:.6f} {(y + h/2)/im['height']:.6f} "
                                     f"{w/im['width']:.6f} {h/im['height']:.6f}")
    (YOLO / split / "images").mkdir(parents=True, exist_ok=True)
    (YOLO / split / "labels").mkdir(parents=True, exist_ok=True)
    for iid, im in imgs.items():
        f = src / im["file_name"]
        dst = YOLO / split / "images" / im["file_name"]
        if not dst.exists():
            os.symlink(f, dst)
        (YOLO / split / "labels" / (pathlib.Path(im["file_name"]).stem + ".txt")).write_text("\n".join(labels[iid]))
    return len(imgs), sum(len(v) for v in labels.values())

for s in ["train", "valid", "test"]:
    print(s, "images, boxes:", convert(s))

## 3. (Optional, recommended) add the team's electrician-tool photos

Label ~150 photos in Label Studio / CVAT / Roboflow with classes `multimeter`, `tester_screwdriver`, `wire_stripper`, `insulated_pliers`, export in **YOLO format**, zip it as `hunar_tools.zip` with `images/` and `labels/` folders whose class ids follow the order below, and upload it to `/content`. Only photos taken **with written consent** (`docs/consent_form.md`) and with faces blurred. Skip this cell if you have none yet.

In [ ]:
TOOL_NAMES = ["multimeter", "tester_screwdriver", "wire_stripper", "insulated_pliers"]
tools_zip = pathlib.Path("/content/hunar_tools.zip")
if tools_zip.exists():
    import random, shutil
    t = pathlib.Path("/content/tools"); zipfile.ZipFile(tools_zip).extractall(t)
    imgs = sorted((t / "images").glob("*"))
    random.Random(0).shuffle(imgs)
    n = len(imgs); parts = {"train": imgs[: int(.7*n)], "valid": imgs[int(.7*n): int(.85*n)], "test": imgs[int(.85*n):]}
    for split, files in parts.items():
        for f in files:
            shutil.copy(f, YOLO / split / "images" / f.name)
            lab = (t / "labels" / (f.stem + ".txt")).read_text().splitlines()
            # shift tool class ids after the PPE classes
            lab = [f"{int(l.split()[0]) + len(NAMES)} {' '.join(l.split()[1:])}" for l in lab if l.strip()]
            (YOLO / split / "labels" / (f.stem + ".txt")).write_text("\n".join(lab))
    NAMES = NAMES + TOOL_NAMES
    print("added", n, "team tool photos; classes:", NAMES)
else:
    print("no team tool photos: training PPE classes only")

## 4. Leakage check: near-duplicate frames between train and test

We hash every image (pHash) and call a test image *leaked* if some training image is within Hamming distance 4. mAP is reported on the full test split **and** on the clean subset without leaked images; the clean number is the honest one.

In [ ]:
import imagehash
from PIL import Image
def phashes(split):
    return {p.name: imagehash.phash(Image.open(p).convert("RGB")) for p in (YOLO / split / "images").iterdir()}
train_h = phashes("train"); test_h = phashes("test")
import numpy as np
T = np.array([h.hash.flatten() for h in train_h.values()])
leaked = []
for name, h in test_h.items():
    d = (T != h.hash.flatten()).sum(1).min()
    if d <= 4:
        leaked.append(name)
print(f"{len(leaked)} of {len(test_h)} test images have a near-duplicate in train ({len(leaked)/len(test_h):.0%})")
clean = YOLO / "test_clean"
for sub in ["images", "labels"]:
    (clean / sub).mkdir(parents=True, exist_ok=True)
for p in (YOLO / "test" / "images").iterdir():
    if p.name not in leaked:
        dst = clean / "images" / p.name
        if not dst.exists():
            os.symlink(os.path.realpath(p), dst)
        lab = YOLO / "test" / "labels" / (p.stem + ".txt")
        (clean / "labels" / lab.name).write_text(lab.read_text())
print("clean test images:", sum(1 for _ in (clean / "images").iterdir()))

## 5. Train YOLOv8n

In [ ]:
import yaml
cfg = {"path": str(YOLO), "train": "train/images", "val": "valid/images", "test": "test/images",
       "names": {i: n for i, n in enumerate(NAMES)}}
yaml.safe_dump(cfg, open(YOLO / "data.yaml", "w"))
cfg_clean = dict(cfg, test="test_clean/images"); yaml.safe_dump(cfg_clean, open(YOLO / "data_clean.yaml", "w"))

from ultralytics import YOLO as Y
# Ultralytics prints the time per epoch after the first epoch: multiply by EPOCHS to estimate the total.
# If time is short, lower EPOCHS (e.g. 30) and say so in the report; the value is saved in the results.
EPOCHS = 50
model = Y("yolov8n.pt")
model.train(data=str(YOLO / "data.yaml"), epochs=EPOCHS, imgsz=640, batch=32, seed=0, deterministic=True,
            project=RUNS, name="hunar_ppe", exist_ok=True)

**Only if Colab disconnected during training:** re-run cells 0 to 2 (mount, download, convert) and
step 4 (leakage), then run this cell instead of the one above. It continues from the last saved epoch.

In [ ]:
# from ultralytics import YOLO as Y
# model = Y(f"{RUNS}/hunar_ppe/weights/last.pt")
# model.train(resume=True)

## 6. Evaluate on the test split and on the clean test split; write `m2_detector.json`

In [ ]:
best = f"{RUNS}/hunar_ppe/weights/best.pt"
m = Y(best)
def evaluate(data_yaml, name):
    r = m.val(data=data_yaml, split="test", imgsz=640, batch=32, plots=True, project=RUNS, name=name, exist_ok=True)
    per_class = {}
    for i, c in enumerate(r.box.ap_class_index):
        p, rc, ap50, ap = r.box.class_result(i)
        per_class[NAMES[int(c)]] = {"precision": round(float(p), 4), "recall": round(float(rc), 4),
                                    "mAP50": round(float(ap50), 4), "mAP50_95": round(float(ap), 4)}
    try:
        cm = r.confusion_matrix.matrix.astype(int).tolist()
    except AttributeError:
        cm = None
    n_imgs = len(list((YOLO / ("test" if name == "eval_test" else "test_clean") / "images").iterdir()))
    return {"n_images": n_imgs, "mAP50": round(float(r.box.map50), 4), "mAP50_95": round(float(r.box.map), 4),
            "precision": round(float(r.box.mp), 4), "recall": round(float(r.box.mr), 4), "per_class": per_class,
            "confusion_matrix": cm}

result = {
    "experiment": "m2_detector", "data_kind": "real_public_dataset",
    "dataset": "keremberke/protective-equipment-detection (Roboflow PPE v7, CC BY 4.0)" +
               (" + team tool photos" if tools_zip.exists() else ""),
    "classes": NAMES, "model": "yolov8n", "epochs": EPOCHS, "imgsz": 640, "seed": 0,
    "ultralytics_version": ultralytics.__version__,
    "command": "notebooks/m2_train_yolo_colab.ipynb on Google Colab", "created": datetime.datetime.now().isoformat(timespec="seconds"),
    "leakage": {"test_images": len(test_h), "near_duplicate_in_train": len(leaked)},
    "test": evaluate(str(YOLO / "data.yaml"), "eval_test"),
    "test_clean": evaluate(str(YOLO / "data_clean.yaml"), "eval_test_clean"),
    "note": "Presence detection only; the model does not judge workmanship quality.",
}
json.dump(result, open("/content/m2_detector.json", "w"), indent=2)
json.dump(result, open(f"{RUNS}/m2_detector.json", "w"), indent=2)  # second copy next to the weights
print(json.dumps({k: result[k] for k in ["leakage"]}, indent=1))
print("test     mAP50 %.3f  mAP50-95 %.3f" % (result["test"]["mAP50"], result["test"]["mAP50_95"]))
print("clean    mAP50 %.3f  mAP50-95 %.3f" % (result["test_clean"]["mAP50"], result["test_clean"]["mAP50_95"]))

## 7. Bring the results back to the repo

1. Download `/content/m2_detector.json` → put it in `experiments/m2_detector.json`.
2. Download `best.pt` (from `RUNS/hunar_ppe/weights/`) → save as `data/models/hunar_ppe_tools_yolov8n.pt` (gitignored).
3. Download `confusion_matrix.png` (from `RUNS/eval_test_clean/`) → `docs/figures/m2_confusion_matrix.png`.
4. On the laptop: `uv sync --extra vision && make results` — the API now reports the detector as available and the README/report tables pick up the numbers.

In [ ]:
from google.colab import files
files.download("/content/m2_detector.json")
files.download(f"{RUNS}/hunar_ppe/weights/best.pt")
files.download(f"{RUNS}/eval_test_clean/confusion_matrix.png")